### [ 주제 : 판정 로그 DB (MySQL) ]
- **[0] 접속** : db_config.py에서 정보 읽어 MySQL 연결
- **[1] 테이블 생성** : judgment_log
- **[2] 판정 기록** : INSERT
- **[3] 조회** : 설비별 이상 횟수

> **왜 DB?** 소리 파일이 아니라 **판정 이력**(언제·어느 설비·오차·판정)을 쌓아 설비별 추이를 조회하기 위함.
> 큰 소리는 파일, 조회할 이력은 DB — 실무 표준 분리.
>
> **준비물**: `db_config.py`에 비밀번호 입력됨 / MySQL 서버 실행 중 / `sound_anomaly` DB 생성됨 (접속 테스트에서 만듦)

In [12]:
## [0] 접속
import sys
sys.path.insert(0,'..')
import pymysql
from db_config import DB_CONFIG
conn = pymysql.connect(**DB_CONFIG)
cur = conn.cursor()
print('접속 완료')

접속 완료


In [13]:
## [1] 판정 로그 테이블 생성
cur.execute('''CREATE TABLE IF NOT EXISTS
judgment_log(
                id          INT AUTO_INCREMENT
                PRIMARY KEY,
                created_at  DATETIME,
                machine_id  VARCHAR(20),
                error       FLOAT,
                judgment   VARCHAR(20)               
                )''')
conn.commit()




In [14]:
## [2] 판정 1건 저장 (MySQL은 자리표시자가 %s)
from datetime import datetime

cur.execute('INSERT INTO judgment_log (created_at, machine_id, error, judgment) VALUES (%s, %s, %s, %s)',
            (datetime.now(), 'id_00', 0.0123, 'abnormal'))
conn.commit()

print(cur.rowcount, '건 저장됨')

1 건 저장됨


In [15]:
## [3] 조회 : 설비별 이상 판정 횟수
cur.execute('''SELECT machine_id, COUNT(*) FROM judgment_log WHERE judgment = 'abnormal' GROUP BY machine_id ''')

for row in cur.fetchall():
    print(row)



('id_00', 2)


In [16]:
## [4] 연결 닫기
conn.close()


In [17]:
## 데모용: 학습 때 쓴 정규화 기준(MIN/MAX)을 파일로 저장
import numpy as np

SAVE_DIR = '../data/processed/slider'
normalNP = np.load(f'{SAVE_DIR}/id_00_normal.npy')

np.random.seed(42)                              # 03과 똑같은 분리 재현
idx = np.random.permutation(len(normalNP))
TRAIN_NUM = int(len(normalNP) * 0.8)
trainNP = normalNP[idx[:TRAIN_NUM]]

MIN = trainNP.min()
MAX = trainNP.max()
np.save('../models/minmax.npy', np.array([MIN, MAX]))   # 모델 옆에 저장
print('저장 완료 :', MIN, MAX)


저장 완료 : -77.88751 7.1573534


## 다음 단계

- 실제로는 04의 판정 결과(복원 오차, normal/abnormal)를 이 테이블에 INSERT하면 됨
- 다음: `app.py` (Streamlit 데모) — wav 올리면 판정하고, 그 결과를 여기 DB에 기록
